In [36]:
import re
import pandas as pd
import numpy as np

years_annual = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]  #cn.years_annual

##### Functions to carry over to geospatial implementation

In [37]:
""" Regex-based land-use reclassification rules
These rules replace the default land use classes.
1. Convert annual GLAD LC values to LU tokens.
2. Use regex to identify token patterns for exceptions.
3. Reclassify token arrays and assign a matching node_code array.

Node codes used here:
1) Settlements and Infrastructure:
    10 = Built from GLAD data

2) Cropland:
    20  = Crop from GLAD data
    21  = Crop from oil palm extent
    22  = Crop from SDPT tree crop extent
    23X = Crop from permanent agriculture driver
         - 233 = TV after TCL where driver is permanent agriculture (assume tree crops)
         - 234 = SV after TCL outside GPW extent (i.e. "rangeland") where driver is permanent agriculture (assume crops)

3) Forest:
    30  = Tall veg from GLAD data
    31  = Forest from SDPT planted forest extent
    32  = Forest from GMW mangrove extent
    33X = Short vegetation or bare reclassified as Forest using drivers rules (assume unstocked forest)
        333 = Forest from shifting cultivation driver
        334 = Forest from logging driver
        335 = Forest from wildfire driver
        337 = Forest from natural disturbance driver

4) Grassland:
    40 = Short veg from GLAD data
    41 = Short veg from permanent agriculture driver
        - SV after TCL inside GPW extent where driver is permanent agriculture (assume rangeland)


5) Wetland:
    50 = Wetland from GLAD data

6) Other
    60 = Bare from GLAD data
    61 = Water from GLAD data
    62 = Snow/ice from GLAD data
"""

# Settlements > Cropland > Forest Land > Grassland > Wetlands > Other
# Default GLAD LC numeric values
settlement_lc   = {250}                                         # Built up
cropland_lc     = {244}                                         # Cropland
forest_lc       = set(range(27, 49)) | set(range(127, 149))     # Tall vegetation
grass_lc        = set(range(5, 27)) | set(range(105, 127))      # Short veg
wetland_lc      = set(range(200, 205))                          # Wetland
bare_lc         = set(range(0, 5)) | set(range(100, 105))       # Bare
water_lc        = set(range(205, 208)) | {254}                  # Open water
ice_lc          = {241}                                         # Snow/ice

# Lookup table to go from GLAD LC code -> default LU token
lc_token_map = {
    **{v: "S" for v in settlement_lc},
    **{v: "C" for v in cropland_lc},
    **{v: "F" for v in forest_lc},
    **{v: "G" for v in grass_lc},
    **{v: "W" for v in wetland_lc},
    **{v: "B" for v in bare_lc},
    **{v: "O" for v in water_lc},
    **{v: "I" for v in ice_lc},
}

# Function to get land use token per land cover numeric value (tokens used for regex exception rules)
def token_for_lc(v):
    if v not in lc_token_map:
        raise ValueError(f"Unknown GLCLU code: {v}")
    return lc_token_map[v]

# Node code values based on what exception was applied
node_code_map = {
    "built_glad": 10,

    "crop_glad": 20,
    "crop_oil_palm": 21,
    "crop_sdpt_tree_crop": 22,
    "crop_perm_ag_driver": 23,

    "forest_glad": 30,
    "forest_gmw_mangrove": 31,
    "forest_sdpt_planted_forest": 32,
    "forest_shift_cult_driver": 333,
    "forest_logging_driver": 334,
    "forest_wildfire_driver": 335,
    "forest_nat_dist_driver": 337,
    "forest_glad_majority_years": 34,
    "forest_veg_bare_sett_mix": 35,
    "forest_veg_bare_crop_mix": 36,
    "forest_tall_short_mix": 37,
    "forest_veg_water_mix": 38,
    "forest_unstocked_pre_oil_palm": 39, 

    "grass_glad": 40,
    "grass_gpw": 41,
    "grass_hard_commod_driver": 422,
    "grass_settlement_driver": 426,
    "grass_unknown_driver": 420,
    "grass_glad_majority_years": 43,
    "grass_veg_bare_sett_mix": 44,
    "grass_veg_bare_crop_mix": 45,
    "grass_tall_short_mix": 46,
    "grass_veg_water_mix": 47,


    "wetland_glad": 50,
    "wetland_glad_majority_years": 51,
    "wetland_veg_water_mix": 52,

    "bare_glad": 60,
    "bare_glad_majority_years": 61,

    "water_glad": 70,
    "water_glad_majority_years": 71,
    "water_veg_water_mix": 72,

    "ice_glad": 80,
}

# Default node codes before rules are applied
def default_node_code(token):
    if token == "S":
        return node_code_map["built_glad"]
    if token == "C":
        return node_code_map["crop_glad"]
    if token == "F":
        return node_code_map["forest_glad"]
    if token == "G":
        return node_code_map["grass_glad"]
    if token == "W":
        return node_code_map["wetland_glad"]
    if token == "B":
        return node_code_map["bare_glad"]
    if token == "O":
        return node_code_map["water_glad"]
    if token == "I":
        return node_code_map["ice_glad"]
    return None

# Function to override default values based on regex rules
def set_tokens(tokens, node_codes, indices, new_token, node_code):
    for i in indices:
        tokens[i] = new_token
        node_codes[i] = node_code

# Converts char tokens to final int values in LU map
lu_token_map = {
    "S": 1,
    "C": 2,
    "F": 3,
    "G": 4,
    "W": 5,
    "B": 6,
    "O": 6,
    "I": 6,
}

In [ ]:
def planting_idx(planting_year):
    if planting_year <= min(years_annual):
        return 0
    if planting_year > max(years_annual):
        return None
    return int(planting_year - min(years_annual))


def tcl_prior_to_planting(tcl_year, planting_year):
    n_years = 5     # number of years between TCL and oil palm planting year allowed to be considered forest -> cropland conversion
    return (tcl_year > 0 and planting_year > 0 and planting_year - n_years <= tcl_year)


def has_planting_transition(lu_dict):
    planting_year = lu_dict.get("planting_year", 0)
    return (planting_year > min(years_annual) and planting_year <= max(years_annual))

In [ ]:
def check_single_lu_transition(lu_dict, lu_ts):
    transition_count = sum(lu_ts[i] != lu_ts[i - 1] for i in range(1, len(lu_ts)))

    if transition_count > 1:
        debug_info = {
            k: v for k, v in lu_dict.items()
            if k not in {"node_codes"}
        }

        raise ValueError(
            f"More than one LU transition detected.\n"
            f"transition_count: {transition_count}\n"
            f"lu_ts: {lu_ts}\n"
            f"debug_info: {debug_info}"
        )

In [38]:
def apply_extent_rules(lu_dict):
    tokens = lu_dict["tokens"]
    node_codes = lu_dict["node_codes"]

    crop_reclass_idx = [i for i, token in enumerate(tokens) if token in {"F", "G", "W", "B"}]
    forest_reclass_idx = [i for i, token in enumerate(tokens) if token in {"G", "W", "B"}]
    # TODO: May want to consider not including wetland?

    # Get oil palm planting year
    crop_extent = lu_dict["sdpt_tree_crop"] or lu_dict["sdpt_oil_palm"]
    planting_year = lu_dict.get("planting_year", 0)
    planting_later = planting_year > min(years_annual)

    # Crop is highest priority and extents are applied in this order: oil palm -> SDPT tree crop --> pre-2000 plantation
    if lu_dict["pre_2000_plantation"]:
        set_tokens(tokens, node_codes, crop_reclass_idx, "C", node_code_map["crop_oil_palm"])
        return True
    if crop_extent and planting_later:
        return False    # Doesn't apply oil palm exception if planting year hasn't happened yet
    if crop_extent:
        node_code = node_code_map["crop_oil_palm"] if lu_dict["sdpt_oil_palm"] else node_code_map["crop_sdpt_tree_crop"]
        set_tokens(tokens, node_codes, crop_reclass_idx, "C", node_code)
        return True

    # If no crop extent applies, forest extents are applied by this order: GMW mangrove -> SDPT planted forest
    if lu_dict["gmw_mangrove"]:
        set_tokens(tokens, node_codes, forest_reclass_idx, "F", node_code_map["forest_gmw_mangrove"])
        return True
    if lu_dict["sdpt_planted_forest"]:
        set_tokens(tokens, node_codes, forest_reclass_idx, "F", node_code_map["forest_sdpt_planted_forest"])
        return True
    return False

In [ ]:
# Tall vegetation all years
def apply_all_tall_veg(lu_dict):
    tcl_prior = lu_dict["tcl_prior"]
    driver = lu_dict["driver"]

    tokens = lu_dict["tokens"]
    node_codes = lu_dict["node_codes"]
    all_idx = range(len(tokens))

    # If TCL has occurred by the start of timeseries and the driver is permanent ag, assume tall veg is tree crops
    if tcl_prior and driver == 1:
        set_tokens(tokens, node_codes, all_idx, "C", node_code_map["crop_perm_ag_driver"])

    # # If oil palm planting year in interval, allows for F -> C transitions assuming establishment of tree crops
    # if has_planting_transition(lu_dict):
    #     idx = planting_idx(lu_dict["planting_year"])
    #     set_tokens(tokens, node_codes, range(0, idx), "F", node_code_map["forest_glad"])
    #     set_tokens(tokens, node_codes, range(idx, len(tokens)), "C", node_code_map["crop_oil_palm"])
    #     return
    #TODO: Do we want to allow for this kind of transition?


In [40]:
# Short vegetation all years
def apply_all_short_veg(lu_dict):
    tcl_prior = lu_dict["tcl_prior"]
    driver = lu_dict["driver"]
    planting_year = lu_dict["planting_year"]

    tokens = lu_dict["tokens"]
    node_codes = lu_dict["node_codes"]
    all_idx = range(len(lu_dict["tokens"]))

    driver_to_forest_node = {
        3: node_code_map["forest_shift_cult_driver"],
        4: node_code_map["forest_logging_driver"],
        5: node_code_map["forest_wildfire_driver"],
        7: node_code_map["forest_nat_dist_driver"],
    }
    
    # If TCL has occurred by the start of the timeseries and the driver is permanent ag and not in cultivated grass extent, assume crop the entire timeseries
    if tcl_prior and driver == 1:
        if not lu_dict["gpw_cultiv_grass"]:
            set_tokens(tokens, node_codes, all_idx, "C", node_code_map["crop_perm_ag_driver"])
        else:
            set_tokens(tokens, node_codes, all_idx, "G", node_code_map["grass_gpw"])

    # Ig oil palm planting year occurs during interval, assume G -> C transition
    if has_planting_transition(lu_dict):
        idx = planting_idx(planting_year)
        set_tokens(tokens, node_codes, range(0, idx), "G", node_code_map["grass_glad"])
        set_tokens(tokens, node_codes, range(idx, len(tokens)), "C", node_code_map["crop_oil_palm"])
        return
    #TODO: Do we ant to consider F -> C transition if TCL up to 5 years prior to oil palm planting
    
    # If TCL has occurred by the start of the timeseries and the driver is shifting cultivation, logging, wildfire, or other natural disturbances, assume unstocked forest the entire timeseries
    if tcl_prior and driver in driver_to_forest_node:
        set_tokens(tokens, node_codes, all_idx, "F", driver_to_forest_node[driver])


In [ ]:
# Mix of short veg and bare
# Only considered a LU transition if initial landcover >= 3 consecutive years and final land cover >= 3 consecutive years and only 1 transition (i.e. GGGBBBBBBB OR BBBBGGGGGG)
def apply_short_bare(lu_dict):
    tokens = lu_dict["tokens"]
    node_codes = lu_dict["node_codes"]

    token_seq = "".join(tokens)
    all_idx = range(len(tokens))

    # Single transition where initial and final landcover >= 3 consecutive years
    if re.fullmatch(r"(G{3,}B{3,}|B{3,}G{3,})", token_seq):
        return

    # Otherwise collapse to majority class across all years
    g_count = tokens.count("G")
    b_count = tokens.count("B")

    # If G and B have the same number of years, assume G
    if g_count >= b_count:
        set_tokens(tokens, node_codes, all_idx, "G", node_code_map["grass_glad_majority_years"])
    else:
        set_tokens(tokens, node_codes, all_idx, "B", node_code_map["bare_glad_majority_years"])

In [ ]:
# Mix of short veg and tall veg
def apply_tall_short(lu_dict):
    tcl_prior = lu_dict["tcl_prior"]
    tcl_year = lu_dict["tcl_year"]
    tcl_any = tcl_year > 0
    driver = lu_dict["driver"]
    planting_year = lu_dict["planting_year"]

    tokens = lu_dict["tokens"]
    node_codes = lu_dict["node_codes"]

    token_seq = "".join(tokens)
    all_idx = range(len(tokens))

    driver_to_forest_node = {
        3: node_code_map["forest_shift_cult_driver"],
        4: node_code_map["forest_logging_driver"],
        5: node_code_map["forest_wildfire_driver"],
        7: node_code_map["forest_nat_dist_driver"],
    }

    driver_to_grass_node = {
        2: node_code_map["grass_hard_commod_driver"],
        6: node_code_map["grass_settlement_driver"],
    }

    # 1) Check if oil palm planting year occurs during interval (regardless of driver + TCL)
    if has_planting_transition(lu_dict):

        # If oil palm planting year in interval, use the first F -> G transition. Else, use oil palm planting year
        transition_match = re.search(r"F+G", token_seq)
        if transition_match:
            idx = transition_match.end() - 1
        else:
            idx = planting_idx(planting_year)
        pre_plant_tokens = tokens[:idx]

        # If F present before transition or TCL within 5 years before planting, consider it F -> C
        forest_before_planting = ("F" in pre_plant_tokens or tcl_prior_to_planting(tcl_year, planting_year))
        if forest_before_planting:
            pre_token = "F"
            pre_node = node_code_map["forest_unstocked_pre_oil_palm"]
        else:
            pre_token = "G"
            pre_node = node_code_map["grass_glad"]

        set_tokens(tokens, node_codes, range(0, idx), pre_token, pre_node)
        set_tokens(tokens, node_codes, range(idx, len(tokens)), "C", node_code_map["crop_oil_palm"])
        return

    # 2) If TCL occurred before the timeseries, use permanent agriculture driver to determine LU for all years.
        # If the driver is permanent ag and not in cultivated grass extent, assume crop. Else, assume grass.
    if tcl_prior and driver == 1:
        if not lu_dict["gpw_cultiv_grass"]:
            set_tokens(tokens, node_codes, all_idx, "C", node_code_map["crop_perm_ag_driver"])
        else:
            set_tokens(tokens, node_codes, all_idx, "G", node_code_map["grass_gpw"])
        return

    # 3) If TCL during the timeseries, use permanent agriculture and first F -> G transition to determine LU transitions:
        # If the driver is permanent ag and not in cultivated grass extent, assume F -> C transition. Else, assume F -> G transition.
    if tcl_any and not tcl_prior and driver == 1:
        match = re.search(r"F+G", token_seq)
        if match:
            transition_idx = match.end() - 1
            pre_transition_idx = range(0, transition_idx)
            final_idx = range(transition_idx, len(tokens))

            set_tokens(tokens, node_codes, pre_transition_idx, "F", node_code_map["forest_glad"]) # Note: if not using first F->G transition switch node code to forest_tall_short_mix

            if lu_dict["gpw_cultiv_grass"]:
                set_tokens(tokens, node_codes, final_idx, "G", node_code_map["grass_gpw"])
            else:
                set_tokens(tokens, node_codes, final_idx, "C", node_code_map["crop_perm_ag_driver"])
            return

    # 4) If TCL in any year and driver is temporary, assume forest all years.
        # Temporary drivers are: shifting cultivation, logging, wildfire, and other natural disturbances
    if tcl_any and driver in driver_to_forest_node:
        set_tokens(tokens, node_codes, all_idx, "F", driver_to_forest_node[driver])
        return

    # 5) If TCL during timeseries, for hard commodities, settlements/ infrastructure, and unknown, use F->G where it stays G until the end. There must be at least 3 Fs, and at least 3 consecutive Gs until the end to determine LU transitions:
    if tcl_any and not tcl_prior and driver not in {1, 3, 4, 5, 7}:
        terminal_match = re.search(r"F{3,}G{3,}$", token_seq)

        if terminal_match:
            transition_match = re.search(r"F+G", token_seq)

            if not transition_match:
                set_tokens( tokens, node_codes, all_idx, "F", node_code_map["forest_tall_short_mix"])
                return
            else:
                transition_idx = transition_match.end() - 1
                pre_transition_idx = range(0, transition_idx)
                final_idx = range(transition_idx, len(tokens))

                set_tokens(tokens, node_codes, pre_transition_idx, "F", node_code_map["forest_tall_short_mix"])

                if driver in driver_to_grass_node:
                    grass_node = driver_to_grass_node[driver]
                else:
                    grass_node = node_code_map["grass_unknown_driver"]

                set_tokens(tokens, node_codes, final_idx, "G", grass_node)

                return

    # 6) Otherwise use regex fallback. If no TCL + driver, LU can only be forest or grass.
    f_count = tokens.count("F")
    g_count = tokens.count("G")

    # If there is not at least 3 years F or 3 years G, not enough evidence for a true F/G transition. Use majority land use instead.
    if g_count < 3:
        set_tokens( tokens, node_codes, all_idx, "F", node_code_map["forest_glad_majority_years"])
        return
    elif f_count < 3:
        set_tokens( tokens, node_codes, all_idx, "G", node_code_map["grass_glad_majority_years"])
        return

    # Terminal G phase must start with 3 consecutive Gs, allow at most one F, end on G.
    # Option to set total number of G years in terminal phase to >= #.
    # TODO: GGGFGG and GGFGGG allowed but not GGFGG?
    else:

        # Capture the first valid terminal G phase. Must start with 3 consecutive Gs.
        terminal_match = re.search(r"(?P<g>G{3,}(?:F?G*)?)$", token_seq)

        # If no valid terminal G phase, set all years to F
        if not terminal_match:
            set_tokens(tokens, node_codes, all_idx, "F", node_code_map["forest_tall_short_mix"])
            return

        # Option to make number of Gs in terminal G phase > 3
        terminal_g_start_idx = terminal_match.start("g")
        terminal_g_count = tokens[terminal_g_start_idx:].count("G")
        if terminal_g_count < 3:
            set_tokens(tokens, node_codes, all_idx, "F", node_code_map["forest_tall_short_mix"])
            return

        # If there is a valid terminal G phase, look for the first F->G transition and sets that as the transition year since that is when the majority of emissions will occur in the vegetation model.
        transition_match = re.search(r"F+G", token_seq)

        if not transition_match:
            set_tokens(tokens, node_codes, all_idx, "F", node_code_map["forest_tall_short_mix"])
            return


        transition_idx = transition_match.end() - 1
        pre_transition_idx = range(0, transition_idx)
        final_idx = range(transition_idx, len(tokens))

        set_tokens(tokens, node_codes, pre_transition_idx, "F", node_code_map["forest_glad"]) # Note: if not using first F->G transition switch node code to forest_tall_short_mix
        set_tokens(tokens, node_codes, final_idx, "G", node_code_map["grass_tall_short_mix"])

In [ ]:
# Mix of vegetation and water/wetland
def apply_veg_water(lu_dict):
    tokens = lu_dict["tokens"]
    node_codes = lu_dict["node_codes"]

    token_seq = "".join(tokens)
    all_idx = range(len(tokens))

    veg_tokens = {"F", "G"}
    water_tokens = {"W", "O"}

    veg_count = sum(t in veg_tokens for t in tokens)
    water_count = sum(t in water_tokens for t in tokens)

    f_count = tokens.count("F")
    g_count = tokens.count("G")
    w_count = tokens.count("W")
    o_count = tokens.count("O")

    # If there are <3 vegetation years, collapse to majority water/wetland. Tie goes to wetland.
    if veg_count < 3:
        if w_count >= o_count:
            set_tokens(tokens, node_codes, all_idx, "W", node_code_map["wetland_glad_majority_years"])
        else:
            set_tokens(tokens, node_codes, all_idx, "O", node_code_map["water_glad_majority_years"])
        return

    # If there are <3 water/wetland years, collapse to majority tall/short veg. Tie goes to forest.
    if water_count < 3:
        if f_count >= g_count:
            set_tokens(tokens, node_codes, all_idx, "F", node_code_map["forest_glad_majority_years"])
        else:
            set_tokens(tokens, node_codes, all_idx, "G", node_code_map["grass_glad_majority_years"])
        return

    # Vegetation -> water/wetland transition:
    # 3+ consecutive vegetation years followed by 3+ consecutive water/wetland years until the end.
    transition_match = re.search(r"(?P<veg>[FG]{3,})(?P<water>[WO]{3,})$", token_seq)

    if transition_match:
        transition_idx = transition_match.start("water")

        pre_transition_idx = range(0, transition_idx)
        final_idx = range(transition_idx, len(tokens))

        pre_tokens = tokens[:transition_idx]
        final_tokens = tokens[transition_idx:]

        # Prominent vegetation class: if F > 2, forest; otherwise grass.
        if pre_tokens.count("F") > 2:
            pre_token = "F"
            pre_node = node_code_map["forest_veg_water_mix"]
        else:
            pre_token = "G"
            pre_node = node_code_map["grass_veg_water_mix"]

        # Majority water class: if W > 2, wetland; otherwise water.
        if final_tokens.count("W") > 2:
            final_token = "W"
            final_node = node_code_map["wetland_veg_water_mix"]
        else:
            final_token = "O"
            final_node = node_code_map["water_veg_water_mix"]

        set_tokens(tokens, node_codes, pre_transition_idx, pre_token, pre_node)
        set_tokens(tokens, node_codes, final_idx, final_token, final_node)
        return

   # If enough evidence of both groups (both groups >=3) but no valid transition, consider it wetland all years.
    set_tokens( tokens, node_codes, all_idx, "W", node_code_map["wetland_veg_water_mix"])
    return


In [ ]:
# Mix of tall, short and/or bare -> crop
def apply_veg_bare_crop(lu_dict):
    tokens = lu_dict["tokens"]
    node_codes = lu_dict["node_codes"]
    token_seq = "".join(tokens)

    first_c_idx = token_seq.find("C")

    if first_c_idx == -1:
        return

    if "F" in token_seq[:first_c_idx]:
        pre_token = "F"
        pre_node = node_code_map["forest_veg_bare_crop_mix"]
    elif "G" in token_seq[:first_c_idx]:
        pre_token = "G"
        pre_node = node_code_map["grass_veg_bare_crop_mix"]
    else:
        # pre_token = "B"
        # pre_node = node_code_map["bare_veg_bare_crop_mix"]
        return # Only option left would be all B -> C

    set_tokens(tokens, node_codes, range(0, first_c_idx), pre_token, pre_node)
    set_tokens(tokens, node_codes, range(first_c_idx, len(tokens)), "C", node_code_map["crop_glad"])

In [ ]:
# Mix of tall > short or bare > settlements and infrastructure
def apply_veg_bare_built(lu_dict):
    tokens = lu_dict["tokens"]
    node_codes = lu_dict["node_codes"]
    token_seq = "".join(tokens)

    first_s_idx = token_seq.find("S")

    if first_s_idx == -1:
        return

    if "F" in token_seq[:first_s_idx]:
        pre_token = "F"
        pre_node = node_code_map["forest_veg_bare_sett_mix"]
    elif "G" in token_seq[:first_s_idx]:
        pre_token = "G"
        pre_node = node_code_map["grass_veg_bare_sett_mix"]
    else:
        pre_token = "B"
        pre_node = node_code_map["bare_veg_bare_sett_mix"]

    set_tokens(tokens, node_codes, range(0, first_s_idx), pre_token, pre_node)
    set_tokens(tokens, node_codes, range(first_s_idx, len(tokens)), "S", node_code_map["built_glad"])

In [41]:
def apply_regex_rules(lc_timeseries, driver, tcl_year, pre_2000_plantation, planting_year, sdpt_oil_palm, sdpt_tree_crop, sdpt_planted_forest, gmw_mangrove, gpw_cultiv_grass):

    # Create default token array and default node code array from LC timeseries
    tokens = [token_for_lc(v) for v in lc_timeseries]               #char array representing land use timeseries
    node_codes = [default_node_code(token) for token in tokens]     #int array representing class definition rules applied throughout the timeseries

    lu_dict = {
        "tokens": tokens,
        "node_codes": node_codes,
        "driver": driver,
        "tcl_year": tcl_year,
        "tcl_prior": (tcl_year != 0 and tcl_year <= 2015),  # convert to bool #TODO: Don't precalculate?
        "pre_2000_plantation": (pre_2000_plantation == 1),
        "planting_year": planting_year,
        "sdpt_oil_palm": (sdpt_oil_palm == 1),
        "sdpt_tree_crop": sdpt_tree_crop,
        "sdpt_planted_forest": sdpt_planted_forest,
        "gmw_mangrove": gmw_mangrove,
        "gpw_cultiv_grass": gpw_cultiv_grass,
    }

    # Check if oil palm, tree crop or forest based on special cases
    extent_rule_applied = apply_extent_rules(lu_dict)

    if not extent_rule_applied:
        token_seq = "".join(tokens) #Creates a concat string
        if re.fullmatch(r"[FGBS]+", token_seq) and "S" in token_seq and re.search(r"[FGB]", token_seq):
            apply_veg_bare_built(lu_dict)
        elif re.fullmatch(r"[FGBC]+", token_seq) and "C" in token_seq and re.search(r"[FGB]", token_seq):
            apply_veg_bare_crop(lu_dict)
        elif re.fullmatch(r"F+", token_seq):
            apply_all_tall_veg(lu_dict)
        elif re.fullmatch(r"G+", token_seq):
            apply_all_short_veg(lu_dict)
        elif re.fullmatch(r"[FG]+", token_seq):
            apply_tall_short(lu_dict)
        elif re.fullmatch(r"[GB]+", token_seq):
            apply_short_bare(lu_dict)
        elif re.fullmatch(r"[FGWO]+", token_seq) and re.search(r"[FG]", token_seq) and re.search(r"[WO]", token_seq):
            apply_veg_water(lu_dict)


    # Final token and node code timeseries
    final_tokens = lu_dict["tokens"]
    node_code_ts = lu_dict["node_codes"]

    # Convert final tokens to numeric LU codes
    lu_ts = [lu_token_map[token] for token in final_tokens]

    # Check that there is only one land use transition during the timeseries
    check_single_lu_transition(lu_dict, lu_ts)

    # Create transition timeseries: 2015_2016 through 2023_2024
    transition_ts = [int(f"{lu_ts[i]}{lu_ts[i + 1]}") for i in range(len(lu_ts) - 1)]

    # Create sequential unique LU summary ([3, 3, 3, 4, 4, 4, 2, 2, 2] -> [3, 4, 2] -> Forest to Grass to Crop)
    summary = []
    for lu in lu_ts:
        if not summary or lu != summary[-1]:
            summary.append(lu)

    return lu_ts, node_code_ts, transition_ts, summary


##### Functions to run IPCC land use classification on tabular data

In [42]:
# Dictionaries to convert numeric output to text for export csv
driver_code_map = {
    1: "permanent_agriculture",
    2: "hard_commodities",
    3: "shifting_cultivation",
    4: "logging",
    5: "wildfire",
    6: "settlements_infrastructure",
    7: "other_natural_disturbances"
}

lu_code_map = {
    1: "settlements_infrastructure",
    2: "cropland",
    3: "forest",
    4: "grassland",
    5: "wetland",
    6: "other",
}

# Reverse lookup because node_code_map is name -> code, but export needs code -> name.
node_code_text_map = {v: k for k, v in node_code_map.items()}

# Converts input csv values as boolean values for classification rules
def as_bool(v):
    if pd.isna(v):
        return False
    if isinstance(v, str):
        v = v.strip().lower()
        if v in {"", "false", "f", "no", "n", "0"}:
            return False
        if v in {"true", "t", "yes", "y", "1"}:
            return True
        return False
    return bool(v)

# Converts input csv values as int values for classification rules
def as_int_or_0(v):
    if pd.isna(v):
        return int(0)
    try:
        return int(v)
    except (TypeError, ValueError):
        return int(0)

# Converts input csv values as float values for classification rules
def as_float_or_0(v):
    if pd.isna(v):
        return 0.0
    try:
        return float(v)
    except (TypeError, ValueError):
        return 0.0

# Iterate through each scenario and classify LC to create LU timeseries
def classify_dataframe(df):
    out = []
    lc_cols = [f"lc_{y}" for y in years_annual]
    for idx, row in df.iterrows():
        scenario_id = row.get("id", idx)
        lc_ts=[as_int_or_0(row.get(lc)) for lc in lc_cols]        #Array of ints
        driver=as_int_or_0(row.get("driver"))                               #Int
        tcl_year=as_int_or_0(row.get("tcl_year"))                           #Int
        pre_2000_plantation = as_int_or_0(row.get("pre_2000_plantation"))   #Int
        planting_year = as_float_or_0(row.get("planting_year"))               #Float
        sdpt_oil_palm = as_int_or_0(row.get("sdpt_oil_palm"))               #Int
        sdpt_tree_crop=as_bool(row.get("sdpt_tree_crop"))               #Boolean
        sdpt_planted_forest=as_bool(row.get("sdpt_planted_forest"))     #Boolean
        gmw_mangrove=as_bool(row.get("gmw_mangrove"))                   #Boolean
        gpw_cultiv_grass=as_bool(row.get("gpw_cultiv_grass"))           #Boolean

        lu_ts, node_code_ts, transition_ts, summary = apply_regex_rules(lc_ts, driver, tcl_year, pre_2000_plantation, planting_year, sdpt_oil_palm, sdpt_tree_crop, sdpt_planted_forest, gmw_mangrove, gpw_cultiv_grass)

        # Convert numeric values to text values in export columns
        # Land use timeseries
        lu_cols = {f"LU_{y}": lu_code_map.get(lu_ts[i], "unknown") for i, y in enumerate(years_annual)}

        # Node code timeseries
        node_code_cols = {f"node_{y}": node_code_text_map.get(node_code_ts[i], "unknown") for i, y in enumerate(years_annual)}

        # Land use transition timeseries (create column to flag whether a transition happened at all)
        conversion = False  # Land use transition flag
        trans_cols = {}

        for i, (a, b) in enumerate(zip(years_annual[:-1], years_annual[1:])):
            transition_code = int(transition_ts[i])

            from_code = transition_code // 10
            to_code = transition_code % 10

            from_lu = lu_code_map[from_code]
            to_lu = lu_code_map[to_code]

            key = f"LU_{a}_{b}"

            if from_code == to_code:
                trans_cols[key] = f"{from_lu} remaining {to_lu}"
            else:
                trans_cols[key] = f"{from_lu} to {to_lu}"
                conversion = True

        # Summary all LU transitions that occurred during entire timeseries
        summary_text = " to ".join(lu_code_map.get(lu, str(lu))for lu in summary)

        out.append({
            "id": scenario_id,
            "tcl_year": (np.nan if tcl_year == 0 else tcl_year),
            "driver": (np.nan if driver == 0 else driver_code_map.get(driver, str(driver))),
            "pre_2000_plantation": True if pre_2000_plantation else np.nan,
            "planting_year": np.nan if planting_year == 0 else planting_year,
            "sdpt_oil_palm": True if sdpt_oil_palm else np.nan,
            "sdpt_tree_crop": True if sdpt_tree_crop else np.nan,
            "sdpt_planted_forest": True if sdpt_planted_forest else np.nan,
            "gmw_mangrove": True if gmw_mangrove else np.nan,
            "gpw_cultiv_grass": True if gpw_cultiv_grass else np.nan,
            **lu_cols,
            **node_code_cols,
            **trans_cols,
            "summary": summary_text,
            "conversion_occurred": conversion,
        })

    return pd.DataFrame(out)

##### Read in scenarios from spreadsheet and export the resulting land use classification spreadsheet

In [44]:
# Read in data
file_path = "/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/scripts/postprocessing/LUC/conversion_LUC_scenarios.xlsx"
sheet_name = "scenarios"
scenarios_df = pd.read_excel(file_path, sheet_name=sheet_name)

# Coerce to numeric
lc_cols = [f"lc_{y}" for y in years_annual]
numeric_cols = lc_cols + ["driver", "tcl_year"]
for c in numeric_cols:
    if c in scenarios_df.columns:
        scenarios_df[c] = pd.to_numeric(scenarios_df[c], errors="coerce")

# Run land use classification
results_df = classify_dataframe(scenarios_df)

# Export results to xlsx
results_df.to_excel("/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/scripts/postprocessing/LUC/conversion_LUC_scenario_results.xlsx", index=False)

# Print results
print("\nClassification Results:")
print(results_df)


Classification Results:
      id  tcl_year                      driver oil_palm sdpt_tree_crop  \
0    1.0       NaN                         NaN     True            NaN   
1    2.0       NaN                         NaN      NaN           True   
2    3.0       NaN                         NaN      NaN            NaN   
3    4.0       NaN                         NaN      NaN            NaN   
4    5.0    2015.0       permanent_agriculture      NaN            NaN   
5    6.0    2015.0       permanent_agriculture      NaN            NaN   
6    7.0    2020.0       permanent_agriculture      NaN            NaN   
7    8.0    2020.0       permanent_agriculture      NaN            NaN   
8    9.0    2015.0        shifting_cultivation      NaN            NaN   
9   10.0    2015.0                     logging      NaN            NaN   
10  11.0    2015.0                    wildfire      NaN            NaN   
11  12.0    2015.0  other_natural_disturbances      NaN            NaN   
12  13.0    2